# RVC — Current Google Colab (Python 3.13)
GPUランタイムを選び、上から順に実行してください。現在のColabのTorch/CUDAを保ったまま、専用環境に追加依存を入れます。
[セットアップ説明](https://github.com/yosi2112/Retrieval-based-Voice-Conversion-WebUI/blob/codex/colab-python313/docs/jp/colab.md)


In [ ]:
import importlib.metadata as metadata
import json, sys, subprocess
from pathlib import Path

print(sys.version)
print(json.dumps({name: metadata.version(name) for name in
    ("torch", "torchaudio", "numpy", "librosa", "gradio", "transformers")}, indent=2))
if sys.version_info[:2] != (3, 13):
    raise RuntimeError("最新のPython 3.13ランタイムを選択してください。")
subprocess.run(["nvidia-smi"], check=True)


In [ ]:
REPO_URL = "https://github.com/yosi2112/Retrieval-based-Voice-Conversion-WebUI.git"
REPO_REF = "codex/colab-python313"
PROJECT = Path("/content/Retrieval-based-Voice-Conversion-WebUI")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(PROJECT)], check=True)
elif not (PROJECT / "tools/colab_setup.py").is_file():
    raise RuntimeError("既存のリポジトリが旧版です。更新版を別フォルダーへ取得するか、変更を保全してから更新してください。")
subprocess.run([sys.executable, str(PROJECT / "tools/colab_setup.py")], cwd=PROJECT, check=True)
PYTHON = str(PROJECT / ".venv-colab/bin/python")


## モデルの取得
公開RVCモデルをダウンロードします。すでにあるファイルはダウンロードキャッシュを再利用します。


In [ ]:
download_code = r"""
from pathlib import Path
from huggingface_hub import snapshot_download
import zipfile
root = Path.cwd()
snapshot_download("lj1995/VoiceConversionWebUI", allow_patterns=["hubert_base/*", "pretrained/*", "pretrained_v2/*", "pymss_weights/*"], local_dir=str(root / "assets"))
snapshot_download("lj1995/VoiceConversionWebUI", allow_patterns=["rmvpe.pt"], local_dir=str(root / "assets/rmvpe"))
snapshot_download("lj1995/VoiceConversionWebUI", allow_patterns=["mute.zip"], local_dir=str(root / ".model-downloads"))
logs = root / "logs"
logs.mkdir(exist_ok=True)
with zipfile.ZipFile(root / ".model-downloads/mute.zip") as archive:
    for name in archive.namelist():
        if not (logs / name).resolve().is_relative_to(logs.resolve()):
            raise RuntimeError("Invalid archive path")
    archive.extractall(logs)
"""
subprocess.run([PYTHON, "-c", download_code], cwd=PROJECT, check=True)
subprocess.run([PYTHON, "tools/colab_check.py", "--require-cuda", "--require-models"], cwd=PROJECT, check=True)


## WebUIを起動
起動ログに表示されるGradioリンクを開きます。学習・特徴量抽出も同じPythonで実行します。


In [ ]:
subprocess.run([PYTHON, "webui.py", "--colab", "--pycmd", PYTHON], cwd=PROJECT, check=True)
